<a href="https://colab.research.google.com/github/michalis0/MGT-502-Data-Science-and-Machine-Learning/blob/main/07_Gen-AI/3-fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center"> LAB - GPT</h1>



### LINKS AND RESOURCES
- https://developers.openai.com/api/docs/guides/supervised-fine-tuning

## INTRODUCTION

A language model is an AI system that can understand and generate human language. It predicts likely next tokens based on context, which allows it to answer questions, summarize text, classify content, and perform many other language tasks.

GPT (Generative Pre-trained Transformer) models are trained on large-scale text data using transformer neural networks, which are designed to handle sequences such as natural language.

Because of this broad pretraining, GPT models can perform a wide range of tasks, including text generation, rewriting, extraction, and reasoning-style assistance.

At the same time, base models are general-purpose. They may not follow a specific style, format, or domain requirement perfectly out of the box.

Fine-tuning helps adapt a base model to your target use case by training it on high-quality examples in your desired format (for example, `system`/`user`/`assistant` message patterns). This typically improves consistency, formatting, and domain-specific behavior for that task.

### What is fine tuning?
Fine-tuning is a process of retraining a pre-trained model on a specific task or domain. The idea behind fine-tuning is to leverage the pre-existing knowledge of a pre-trained model and apply it to a more specific task. Fine-tuning allows you to customize a pre-trained model to your particular needs, which can improve its performance on the given task.

### Outline
1. Get OpenAI API key

2. Create training data

3. Check the training data

4. Upload training data

5. Fine-tune model

6. Check fine-tune progress

7. Save fine-tuned model

8. Test the new model on a new prompt

In [ ]:
import json
from openai import OpenAI

## 1. OpenAI Key

In [35]:
api_key ="YOUR_OPENAI_API_KEY"
client = OpenAI(api_key=api_key)

## 2. Create Training Data

Use chat-format training examples with a `messages` array (`system`, `user`, `assistant`) for modern fine-tuning jobs.

In [34]:
data_file = [
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Prompt"},
            {"role": "assistant", "content": "Ideal answer."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Prompt"},
            {"role": "assistant", "content": "Ideal answer."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Prompt"},
            {"role": "assistant", "content": "Ideal answer."}
        ]
    }
]

## 3. Save dict as JSONL file

In [ ]:
file_name = "training_data.jsonl"

with open(file_name, 'w') as outfile:
    for entry in data_file:
        json.dump(entry, outfile)
        outfile.write('\n')

## 4. Upload file to your OpenAI account

In [ ]:
with open(file_name, "rb") as f:
    upload_response = client.files.create(
        file=f,
        purpose="fine-tune",
    )

upload_response

### Save file name

In [ ]:
file_id = upload_response.id
file_id

## 5. Fine-tune a model

Choose a fine-tunable base model supported in your account (for example, `gpt-4o-mini-2024-07-18`).

In [ ]:
model = "gpt-4o-mini-2024-07-18"
fine_tune_response = client.fine_tuning.jobs.create(
    training_file=file_id,
    model=model,
 )
fine_tune_response

## 6. Check the status of the fine-tuning process

The fine_tune_response refers to the response object obtained after initiating the fine-tuning process. This code snippet allows you to monitor and track the progress and status of the fine-tuning job by examining the events associated with it. The fine_tune_events variable will contain the list of events retrieved.

In [ ]:
fine_tune_events = client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=fine_tune_response.id
)
fine_tune_events

By executing this code, you can obtain the current status, configuration, and other relevant details of the fine-tuning process. The retrieve_response variable will contain the response object with the retrieved information.

In [ ]:
retrieve_response = client.fine_tuning.jobs.retrieve(fine_tune_response.id)
retrieve_response

## 7. Save the fine-tuned model

#### Troubleshooting `fine_tuned_model` as null

During fine-tuning, `fine_tuned_model` may be `None` in the initial response returned by `client.fine_tuning.jobs.create(...)`.

To check status, call `client.fine_tuning.jobs.retrieve(fine_tune_response.id)` and inspect the returned job fields.

You can also list jobs with `client.fine_tuning.jobs.list()` and find the latest job with a non-null `fine_tuned_model`.

Once training completes, `fine_tuned_model` will contain the model ID to use for inference.

### Option 1

In [ ]:
# If fine_tune_response.fine_tuned_model is available, use it
if fine_tune_response.fine_tuned_model is not None:
    fine_tuned_model = fine_tune_response.fine_tuned_model

### Option 2

In [ ]:
# If fine_tune_response.fine_tuned_model is None, list jobs and take the first available fine_tuned_model
if fine_tune_response.fine_tuned_model is None:
    fine_tune_list = client.fine_tuning.jobs.list()
    fine_tuned_model = next((job.fine_tuned_model for job in fine_tune_list.data if job.fine_tuned_model), None)

### Option 3

In [ ]:
# If fine_tune_response.fine_tuned_model is None, retrieve the fine-tuning job and get fine_tuned_model
if fine_tune_response.fine_tuned_model is None:
    fine_tuned_model = client.fine_tuning.jobs.retrieve(fine_tune_response.id).fine_tuned_model

## 8. Test the fine-tuned model

In [ ]:
new_prompt = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Prompt"} 
]

In [ ]:
answer = client.responses.create(
    model=fine_tuned_model,
    input=new_prompt,
    max_output_tokens=100,  # Increase for longer responses
    temperature=0,
 )
answer.output_text